# PatchCore Review Notebook (WideResNet50-2 Weighted Sweep, x224)

This is the curated review notebook for the saved weighted WRN `x224` sweep.

This branch only has the aggregate sweep table checked in, so the notebook focuses on comparing the saved configurations rather than replaying per-variant score distributions.

## Submission Context

- Dataset notebook: `data/dataset/x224/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x224/benchmark_50k_5pct/data_config.toml`
- Artifact root: `experiments/anomaly_detection/patchcore/wideresnet50/x224/weighted/artifacts/patchcore-wideresnet50-weighted`
- Mode: sweep-table review only
- Checkpoint status: no per-variant checkpoints or score CSVs were checked in

## Imports and Sweep Loading

This cell loads the saved weighted sweep table and the best saved row from the checked-in summary.

In [ ]:
try:
    from pathlib import Path
    import json
    import sys
    import matplotlib.pyplot as plt
    import pandas as pd
    from IPython.display import display
    cwd = Path.cwd().resolve()
    candidate_roots = [cwd, *cwd.parents]
    REPO_ROOT = None
    for candidate in candidate_roots:
        if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None:
        raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
    ARTIFACT_ROOT = REPO_ROOT / 'experiments/anomaly_detection/patchcore/wideresnet50/x224/weighted/artifacts/patchcore-wideresnet50-weighted'
    RESULTS_DIR = ARTIFACT_ROOT / 'results'
    PLOTS_DIR = ARTIFACT_ROOT / 'plots'
    sweep_results_df = pd.read_csv(RESULTS_DIR / 'patchcore_sweep_results.csv')
    sweep_summary = json.loads((RESULTS_DIR / 'patchcore_sweep_summary.json').read_text(encoding='utf-8'))
    best_variant = sweep_summary['best_variant']
    display(sweep_results_df.head())
    pd.Series(best_variant)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "from pathlib import path\nimport json\nimport sys\nimport matplotlib.pyplot as plt\nimport pandas as pd\nfrom ipython.display import display\ncwd = path.cwd().resolve()\ncandidate_roots = [cwd, *cwd.parents]\nrepo_root = none\nfor candidate in candidate_roots:\n    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():\n        repo_root = candidate\n        break\nif repo_root is none:\n    raise runtimeerror('could not locate repo root containing src/wafer_defect and configs/')\nartifact_root = repo_root / 'experiments/anomaly_detection/patchcore/wideresnet50/x224/weighted/artifacts/patchcore-wideresnet50-weighted'\nresults_dir = artifact_root / 'results'\nplots_dir = artifact_root / 'plots'\nsweep_results_df = pd.read_csv(results_dir / 'patchcore_sweep_results.csv')\nsweep_summary = json.loads((results_dir / 'patchcore_sweep_summary.json').read_text(encoding='utf-8'))\nbest_variant = sweep_summary['best_variant']\ndisplay(sweep_results_df.head())\npd.series(best_variant)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


In [ ]:
RUN_TRAINING = False
print(f'RUN_TRAINING = {RUN_TRAINING}')


## Weighted Sweep Comparison

These plots compare the saved weight combinations and top-k ratios using the checked-in sweep table.

In [ ]:
try:
    top_plot_df = sweep_results_df.sort_values(['f1', 'auroc'], ascending=False).head(15).copy()
    top_plot_df['label'] = top_plot_df['name']
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].barh(top_plot_df['label'], top_plot_df['f1'], color='#264653')
    axes[0].set_title('Top Weighted Variants by F1')
    axes[0].invert_yaxis()
    axes[1].barh(top_plot_df['label'], top_plot_df['auroc'], color='#2a9d8f')
    axes[1].set_title('Top Weighted Variants by AUROC')
    axes[1].invert_yaxis()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'weighted_top_variants.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    best_by_weight_df = sweep_results_df.sort_values(['weight_name', 'f1', 'auroc'], ascending=[True, False, False]).groupby('weight_name', as_index=False).first().sort_values('f1', ascending=False)
    best_by_weight_df.to_csv(RESULTS_DIR / 'best_by_weight_name.csv', index=False)
    display(best_by_weight_df)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(sweep_results_df['recall'], sweep_results_df['precision'], c=sweep_results_df['f1'], cmap='viridis', s=70)
    for _, row in best_by_weight_df.iterrows():
        ax.text(row['recall'], row['precision'], row['weight_name'], fontsize=8)
    ax.set_xlabel('recall')
    ax.set_ylabel('precision')
    ax.set_title('Weighted Sweep Precision-Recall Tradeoff')
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'weighted_precision_recall_scatter.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "top_plot_df = sweep_results_df.sort_values(['f1', 'auroc'], ascending=false).head(15).copy()\ntop_plot_df['label'] = top_plot_df['name']\nfig, axes = plt.subplots(1, 2, figsize=(15, 6))\naxes[0].barh(top_plot_df['label'], top_plot_df['f1'], color='#264653')\naxes[0].set_title('top weighted variants by f1')\naxes[0].invert_yaxis()\naxes[1].barh(top_plot_df['label'], top_plot_df['auroc'], color='#2a9d8f')\naxes[1].set_title('top weighted variants by auroc')\naxes[1].invert_yaxis()\nplt.tight_layout()\nfig.savefig(plots_dir / 'weighted_top_variants.png', dpi=200, bbox_inches='tight')\nplt.show()\nplt.close(fig)\nbest_by_weight_df = sweep_results_df.sort_values(['weight_name', 'f1', 'auroc'], ascending=[true, false, false]).groupby('weight_name', as_index=false).first().sort_values('f1', ascending=false)\nbest_by_weight_df.to_csv(results_dir / 'best_by_weight_name.csv', index=false)\ndisplay(best_by_weight_df)\nfig, ax = plt.subplots(figsize=(10, 5))\nax.scatter(sweep_results_df['recall'], sweep_results_df['precision'], c=sweep_results_df['f1'], cmap='viridis', s=70)\nfor _, row in best_by_weight_df.iterrows():\n    ax.text(row['recall'], row['precision'], row['weight_name'], fontsize=8)\nax.set_xlabel('recall')\nax.set_ylabel('precision')\nax.set_title('weighted sweep precision-recall tradeoff')\nplt.tight_layout()\nfig.savefig(plots_dir / 'weighted_precision_recall_scatter.png', dpi=200, bbox_inches='tight')\nplt.show()\nplt.close(fig)"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise


## Saved Outputs

This cell summarizes the saved outputs for the weighted sweep review notebook.

In [ ]:
try:
    saved_outputs = {'artifact_root': str(ARTIFACT_ROOT), 'results_dir': str(RESULTS_DIR), 'plots_dir': str(PLOTS_DIR), 'best_variant_name': best_variant['name']}
    saved_outputs
except Exception as exc:
    _codex_msg = str(exc).lower()
    _codex_source = "saved_outputs = {'artifact_root': str(artifact_root), 'results_dir': str(results_dir), 'plots_dir': str(plots_dir), 'best_variant_name': best_variant['name']}\nsaved_outputs"
    _codex_tokens = ('artifact', 'artifacts', 'checkpoint', 'history', 'summary', 'score', 'evaluation', 'umap', 'embedding', 'prediction', 'metadata', 'variant', 'holdout', 'plot', 'result')
    if isinstance(exc, FileNotFoundError):
        print(f'[WARNING] {exc}')
    elif isinstance(exc, NameError):
        print(f'[WARNING] Skipping this cell because earlier artifact-dependent outputs are unavailable: {exc}')
    elif isinstance(exc, (RuntimeError, KeyError, IndexError, ValueError, AttributeError)):
        if any((token in _codex_msg for token in _codex_tokens)) or any((token in _codex_source for token in _codex_tokens)):
            print(f'[WARNING] Skipping this cell because prerequisite artifacts or cached outputs are missing or incomplete: {exc}')
        else:
            raise
    else:
        raise
